# Supplementary: reading an FFN as key-value memory

Companion to [`../02_the_ffn.md`](../02_the_ffn.md) — the exercise from the end of that file, made runnable on a real pretrained model.

**The claim we're testing.** An FFN sublayer is
```
out = W_down · σ(W_up · x)          (row-convention: out = σ(x W_up) W_down)
```
and the key-value reading of it (Geva et al., 2021) says: each of the `d_ff` hidden units is a **memory**. Its `W_up` row is a **key** — a pattern it detects in the incoming residual stream. Its `W_down` column is a **value** — a fixed vector it writes back when the key fires. The nonlinearity is the gate that turns "how well did the key match" into "how much of the value to write."

**What we'll do**, in order:
1. Hook a real FFN and capture its post-activation hidden vector — the vector of key-match scores.
2. Show the write is **concentrated**: on one token, a *single* unit out of 1536 accounts for 81% of the FFN's output.
3. Find the most **selective** units and read their keys off the top-activating contexts.
4. Read their values through the unembedding (**logit lens**) to see which output tokens each value promotes.
5. Check one memory **generalizes** to held-out text, then **ablate** it and watch the prediction it was responsible for collapse.
6. Contrast with attention: the FFN's write **direction** is identical on every input; an attention head's is not.

**Convention note.** Two conventions collide here and it matters for indexing. The doc writes `W_down[i,:]` under the **row-vector** convention (`h W_down`, so `W_down ∈ R^(d_ff × D)`, unit `i` is a *row*). HuggingFace stores `nn.Linear` weights in the **column** convention (`y = W h`), so `down_proj.weight ∈ R^(D × d_ff)` and unit `i`'s value vector is the **column** `down_proj.weight[:, i]`. Everything below uses the HF layout; `[:, i]` is the value vector.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
print(f"torch: {torch.__version__}")

torch: 2.9.1


## Load a small SwiGLU model

`HuggingFaceTB/SmolLM2-135M` — a Llama-architecture model, ~135M params, ~270 MB download. Two reasons it's the right choice here:

- It's a **real pretrained SwiGLU** model (`hidden_act: "silu"`, three FFN matrices), not a GELU two-matrix model. GPT-2, Pythia, GPT-Neo, and TinyStories are all pre-SwiGLU — the exercise's `W_gate`/`W_up`/`W_down` structure doesn't exist in them.
- It **ties embeddings**, so the unembedding `W_U` we need for the logit lens is just `embed_tokens.weight` — no separate head to worry about.

**Run this on CPU in fp32.** The whole notebook needs one forward pass per short string — a fraction of a second on CPU — so MPS buys nothing and costs correctness: FlashAttention-2 is CUDA-only (so `transformers` silently falls back on Apple silicon anyway), `F.scaled_dot_product_attention` on MPS has been unreliable with non-trivial masks and non-fp32 dtypes, and bf16 would make every activation magnitude below noisier than the numbers quoted in this notebook. `attn_implementation="eager"` sidesteps the SDPA path entirely.

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,           # not bf16: keeps activation magnitudes comparable to the notes below
    attn_implementation="eager",   # skip SDPA/flash entirely — matters on MPS, harmless on CPU
).eval()

cfg = model.config
D, d_ff, n_layer = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers
V = cfg.vocab_size

print(f"D (d_model):   {D}")
print(f"d_ff:          {d_ff}   ({d_ff / D:.3f} x D)")
print(f"layers:        {n_layer}")
print(f"vocab:         {V}")
print(f"activation:    {cfg.hidden_act}")
print(f"tied embeddings: {model.lm_head.weight is model.model.embed_tokens.weight}")

D (d_model):   576
d_ff:          1536   (2.667 x D)
layers:        30
vocab:         49152
activation:    silu
tied embeddings: True


## The FFN we're about to instrument

`LlamaMLP` is exactly the SwiGLU block from the doc, with HF's naming:

```
h   = silu(gate_proj(x)) * up_proj(x)      # (B, S, d_ff)  — the key-match scores, gated
out = down_proj(h)                          # (B, S, D)     — the values, summed
```

Note what this model's sizing does: `d_ff = 1536 = 8·576/3` **exactly** — the textbook `8D/3` rule from [`../02_the_ffn.md`](../02_the_ffn.md) self-check #3. So the "three skinny matrices cost the same as two fat ones" arithmetic should come out to the digit here, which the next cell checks. (Not every model does this. Qwen2.5's small models use `d_ff ≈ 5.4D`, and Llama-3-8B uses `3.5D` — both deliberately wider than the rule.)

In [ ]:
mlp = model.model.layers[0].mlp
print(f"{type(mlp).__name__}:")
for name, p in mlp.named_parameters():
    print(f"   {name:20s} {tuple(p.shape)}")

# The 8D/3 arithmetic, on a real checkpoint
swiglu_params  = 3 * D * d_ff
classic_params = 2 * D * (4 * D)
print(f"\nSwiGLU   3 · D · d_ff   = 3 · {D} · {d_ff}  = {swiglu_params/1e6:.6f}M per block")
print(f"classic  2 · D · 4D     = 2 · {D} · {4*D}  = {classic_params/1e6:.6f}M per block")
print(f"identical: {swiglu_params == classic_params}")

# How much of the model is FFN?
ffn_params  = sum(p.numel() for l in model.model.layers for p in l.mlp.parameters())
attn_params = sum(p.numel() for l in model.model.layers for p in l.self_attn.parameters())
total       = sum(p.numel() for p in model.parameters())
print(f"\nFFN:        {ffn_params/1e6:6.1f}M   ({ffn_params/total:.1%} of all params)")
print(f"attention:  {attn_params/1e6:6.1f}M")
print(f"FFN share of transformer-block params: {ffn_params/(ffn_params+attn_params):.1%}")
print(f"embeddings (tied): {V*D/1e6:.1f}M   ({V*D/total:.1%} of all params)")

LlamaMLP:
   gate_proj.weight     (1536, 576)
   up_proj.weight       (1536, 576)
   down_proj.weight     (576, 1536)

SwiGLU   3 · D · d_ff   = 3 · 576 · 1536  = 2.654208M per block
classic  2 · D · 4D     = 2 · 576 · 2304  = 2.654208M per block
identical: True

FFN:          79.6M   (59.2% of all params)
attention:    26.5M
FFN share of transformer-block params: 75.0%
embeddings (tied): 28.3M   (21.0% of all params)


**What you should see:** `gate_proj` and `up_proj` are both `(1536, 576)` — `D → d_ff`; `down_proj` is `(576, 1536)` — `d_ff → D`. The two param counts match exactly at **2.654208M** each: the `⅔` width factor pays for the third matrix to the last digit.

The FFN holds **75% of transformer-block parameters** — the doc's "≈80% of the block" claim, confirmed on a real checkpoint. But only **59% of *all* params**, because at 135M scale the tied embedding matrix (`49152 × 576` = 28.3M) is a hefty **21%** of the model. That fraction shrinks fast with scale: at 8B, `V·D` is a rounding error and the FFN's share of the total approaches its share of the block.

## Capture the post-activation hidden vector

We want `h = silu(gate_proj(x)) * up_proj(x)`, shape `(B, S, d_ff)` — one key-match score per memory per token. There's no need to recompute it: **`h` is exactly the input to `down_proj`**, so a `forward_pre_hook` on `down_proj` hands it to us for free.

This is the general trick for reading intermediates out of someone else's model: don't rewrite `forward`, hook the boundary where the tensor you want is already flowing. `register_forward_pre_hook` sees a module's *inputs*; `register_forward_hook` sees its *outputs*. Always `.detach()` (we're not training) and always `handle.remove()` when done, or the hook fires on every later forward pass and silently leaks memory.

The corpus is 32 short lines chosen for **variety**, not size — facts, code, dates, money, quotes, URLs, plurals — so that different memories get a chance to fire. ~500 tokens is plenty to find strongly selective units.

In [4]:
CORPUS = '''The capital of France is Paris, and the capital of Japan is Tokyo.
The sky is blue, the grass is green, and ripe tomatoes are red.
In 1969 Apollo 11 landed on the Moon; in 1789 the French Revolution began.
Water freezes at 0 degrees Celsius and boils at 100 degrees Celsius.
def add(a, b):
    return a + b
import numpy as np
x = np.zeros((4, 5))
See https://example.com/docs for more information about the API.
Send questions to alice@example.com or bob@example.org.
The cat sat on the mat. The dogs barked at the cars in the street.
She walked to the store, bought some milk, and walked back home.
Shakespeare wrote Hamlet, Macbeth, and King Lear in the early 1600s.
The Pacific Ocean is the largest ocean on Earth, covering 63 million square miles.
My phone number is 555-0132 and my zip code is 94103.
Version 3.11 of Python was released in October 2022.
"Stop right there," she said quietly, "or I will call the police."
The mitochondria produce ATP, the energy currency of the cell.
Oxygen has atomic number 8; carbon has atomic number 6.
Berlin is in Germany, Madrid is in Spain, and Rome is in Italy.
He is taller than his brother, but shorter than his father.
On Monday it rained, on Tuesday it snowed, and by Friday it was sunny.
The company reported revenue of $4.2 billion in the fourth quarter of 2023.
Twelve plus seven equals nineteen, and nineteen minus four equals fifteen.
The children were playing while their parents were cooking dinner.
SELECT name, age FROM users WHERE age > 30 ORDER BY name;
Mount Everest, at 8,849 meters, is the highest mountain in the world.
Dr. Smith arrived at 9:30 a.m. on Sept. 14 for the appointment.
The novel was published in Paris in 1922 by Sylvia Beach.
If it rains tomorrow, then the match will be postponed until Sunday.
Cows eat grass, lions eat meat, and pandas eat bamboo.
The United States has 50 states; Canada has 10 provinces.'''

lines = [l for l in CORPUS.split("\n") if l.strip()]


def capture_ffn_acts(layer_idx, texts):
    '''Run `texts` through the model and collect layer `layer_idx`'s FFN post-activations.

    Returns (A, ids, lens):
      A    (N_tokens, d_ff)  post-activation h for every token, all texts concatenated
      ids  (N_tokens,)       the token id at each row of A
      lens list[int]         token count per text, so we can map a row back to (text, position)
    '''
    grab = {}
    handle = model.model.layers[layer_idx].mlp.down_proj.register_forward_pre_hook(
        lambda mod, args: grab.__setitem__("h", args[0].detach())   # args[0] IS h
    )
    acts, ids = [], []
    with torch.no_grad():
        for t in texts:
            enc = tok(t, return_tensors="pt")
            model(**enc)
            acts.append(grab["h"][0])            # (S, d_ff)
            ids.append(enc["input_ids"][0])      # (S,)
    handle.remove()                              # always clean up
    return torch.cat(acts, 0), torch.cat(ids, 0), [len(i) for i in ids]


LAYER = 25   # a late-but-not-final layer: features are abstract, and values read cleanly as output tokens
A, ids, lens = capture_ffn_acts(LAYER, lines)

# row -> (text index, position within that text), for showing contexts later
owner = [(li, p) for li, n in enumerate(lens) for p in range(n)]

print(f"captured layer {LAYER}: A {tuple(A.shape)}  ({A.shape[0]} tokens x {d_ff} memories)")
print(f"mean |h|: {A.abs().mean():.3f}    max h: {A.max():.2f}    min h: {A.min():.2f}")

captured layer 25: A (507, 1536)  (507 tokens x 1536 memories)
mean |h|: 0.257    max h: 36.50    min h: -46.53


**What you should see:** `A` of shape `(507, 1536)` — 507 tokens, one activation per memory per token. Mean `|h|` ≈ 0.257 but max ≈ 36.5: the distribution is extremely heavy-tailed. That asymmetry is the first hint of the lookup-table picture — almost everything is near zero, and a few things are enormous.

Note the min, ≈ **-46.5**: `h` is **signed**. A gated activation (SiLU/SwiGLU) can go negative, which means a memory can fire with a negative coefficient and write the *negation* of its value vector. A ReLU FFN can only ever add values; a gated one can subtract them too, effectively doubling what each memory can express. Keep that in mind when reading a value's promoted tokens — for a negative `h_i`, the promoted and suppressed lists swap.

## The write is concentrated on a handful of memories

If "lookup table" is the right metaphor, then on any given token most of the FFN's output should come from a *few* fired memories, not a democratic blend of all 1536.

This is easy to check exactly, because the FFN output decomposes into a **sum of per-unit writes** with no cross terms:
```
out = Σ_i  h_i · W_down[:, i]
```
Each unit contributes `h_i` (a scalar) times its own fixed value vector. So we can rank units by the norm of their contribution and ask how much of the total each prefix reproduces. We do this for the `$4.2` token in the revenue sentence.

In [5]:
probe_text = "The company reported revenue of $4.2"

grab = {}
handle = model.model.layers[LAYER].mlp.down_proj.register_forward_pre_hook(
    lambda mod, args: grab.__setitem__("h", args[0].detach()))
with torch.no_grad():
    model(**tok(probe_text, return_tensors="pt"))
handle.remove()

h = grab["h"][0, -1]                                    # (d_ff,) activations on the final token
W_down = model.model.layers[LAYER].mlp.down_proj.weight  # (D, d_ff) — column i is unit i's value vector

contrib = W_down * h                 # (D, d_ff): column i = h_i · W_down[:, i], unit i's actual write
out = contrib.sum(dim=1)             # (D,) the FFN's full write to the residual stream
order = torch.argsort(contrib.norm(dim=0), descending=True)

print(f"prompt: {probe_text!r}   (analyzing its final token, {tok.decode(tok(probe_text)['input_ids'][-1:])!r})")
print(f"full FFN output norm: {out.norm():.1f}\n")
for k in [1, 5, 10, 50, d_ff]:
    partial = contrib[:, order[:k]].sum(dim=1)
    print(f"  top {k:>4} units: {partial.norm()/out.norm()*100:5.1f}% of the output norm, "
          f"cos to full output {F.cosine_similarity(partial, out, dim=0):.3f}")

print(f"\nfraction of the 1536 memories with |h| > 1.0:  {(h.abs() > 1.0).float().mean():.1%}")
print(f"top-5 units by write norm: {order[:5].tolist()}")

prompt: 'The company reported revenue of $4.2'   (analyzing its final token, '2')
full FFN output norm: 256.4

  top    1 units:  81.1% of the output norm, cos to full output 0.763
  top    5 units:  86.6% of the output norm, cos to full output 0.870
  top   10 units:  89.7% of the output norm, cos to full output 0.888
  top   50 units:  96.3% of the output norm, cos to full output 0.940
  top 1536 units: 100.0% of the output norm, cos to full output 1.000

fraction of the 1536 memories with |h| > 1.0:  8.9%
top-5 units by write norm: [1396, 587, 1060, 700, 200]


**What you should see:** on this one token, the **single strongest memory reproduces 81% of the FFN's output norm** (unit 1396), the top 5 give 87%, and the remaining ~1500 units together contribute the last 4%. Fewer than **9%** of memories have `|h| > 1` at all.

Read that carefully, though — 81% of the *norm*, but cosine only 0.763 to the full output. One memory dominates the **magnitude** while the long tail still meaningfully steers the **direction**. That's the honest version of the lookup-table claim: it's "a few loud entries plus a quiet chorus," not "one entry, full stop."

## Find the selective memories, then read their keys

A memory that fires a little on everything is doing something diffuse; a memory that fires *hard on a few tokens and nowhere else* is the interpretable kind. Rank by a crude selectivity score:

```
selectivity_i = max_t h_i(t) / mean_t |h_i(t)|
```

Then for each top unit, print the tokens where it fires hardest along with their left context. **That printout is the key** — you're reading off, empirically, "what pattern does this memory detect."

In [6]:
selectivity = A.max(dim=0).values / (A.abs().mean(dim=0) + 1e-6)
top_units = torch.topk(selectivity, 5).indices.tolist()


def show_key(unit, n_ctx=5, ctx_tokens=9):
    '''Print the contexts where `unit` fires hardest — the empirical read of its key.'''
    col = A[:, unit]
    print(f"unit {unit:>4}   max h = {col.max():6.2f}   selectivity = {selectivity[unit]:.0f}")
    for row in torch.topk(col, n_ctx).indices.tolist():
        li, p = owner[row]
        start = sum(lens[:li])
        seq = ids[start:start + lens[li]]
        left = tok.decode(seq[max(0, p - ctx_tokens):p])
        here = tok.decode(seq[p:p + 1])
        print(f"      {col[row]:6.2f}   ...{left}[[{here}]]")


for u in top_units:
    show_key(u)
    print()

unit 1396   max h =  36.50   selectivity = 65
       36.50   ...The company reported revenue of $4.[[2]]
       31.99   ...The company reported revenue of $[[4]]
       14.08   ...Mount Everest, at [[8]]
       10.72   ...The company reported revenue of[[ $]]
        8.14   ...Mount Everest, at 8,[[8]]

unit  571   max h =  14.51   selectivity = 63
       14.51   ...SELECT name, age FROM users WHERE age >[[ ]]
        3.51   ..., Macbeth, and King Lear in the early[[ ]]
        2.78   ...The novel was published in Paris in 1[[9]]
        1.97   ... Hamlet, Macbeth, and King Lear in the[[ early]]
        1.14   ... in 1789 the French Revolution[[ began]]

unit  577   max h =  26.59   selectivity = 53
       26.59   ...The novel was published in Paris in[[ ]]
       20.96   ... 11 landed on the Moon; in[[ ]]
       19.12   ...4.2 billion in the fourth quarter of[[ ]]
       14.37   ...In[[ ]]
       13.28   ... $4.2 billion in the fourth quarter[[ of]]

unit 1211   max h =  13.66   selec

**What you should see** (the token in `[[ ]]` is the one the memory fired on):

- **unit 1396** fires on `$4.`**`2`**, `$`**`4`**, `at `**`8`** — it detects *"we are inside a large number, right after a currency or magnitude cue."*
- **unit 571** fires on the token after `WHERE age >` and `in the early` — a *"a numeric value belongs here"* slot detector with a syntactic flavor.
- **unit 577** fires on the space right before a year: `published in Paris in`**` `**, `landed on the Moon; in`**` `**, `fourth quarter of`**` `** — *"a year/date is about to start."*
- **unit 1211** fires hard on `On Monday it`**` ra`** and weakly on scattered digits — no clean reading. This is what a **polysemantic** unit looks like.
- **unit 896** fires on every token of the mitochondria sentence — a topic/domain detector rather than a token detector.

Not all of these will be crisp, and that's the honest outcome: some units read as clean rules, some as fuzzy topical ones, some as nothing recognizable. **1396 is the clean one**, so the rest of the notebook follows it.

## Read the values through the unembedding (logit lens)

Now the other half of the exercise. Unit `i`'s value vector `W_down[:, i]` is a direction in the residual stream. Since the residual stream is read at the end by the unembedding `W_U`, we can ask **which output tokens this value promotes** by projecting it:

```
token_scores = (W_down[:, i] ⊙ norm_weight) @ W_Uᵀ          # (V,)
```

Three things to be aware of:

- **`W_U` is `embed_tokens.weight`** here — the model ties embeddings, so the unembedding is the transpose of the input embedding.
- **Folding in the final RMSNorm weight.** The stream passes through `model.norm` before the unembedding, so we multiply elementwise by its learned gain. We *cannot* apply the normalization itself — RMSNorm's rescaling depends on the whole input vector, and we only have this one unit's contribution. So this is an approximation, standard for the logit lens.
- **It's a direction, not a prediction.** We're asking "if this value were written to the stream, which way would it push the output distribution" — skipping the 4 remaining layers that will transform it. The later the layer, the better this approximation holds, which is exactly why we picked layer 25 of 30.

In [7]:
W_U = model.model.embed_tokens.weight     # (V, D) — tied, so this is also the unembedding
norm_w = model.model.norm.weight          # (D,) final RMSNorm gain


def show_value(unit, n_pos=8, n_neg=4):
    '''Project unit `unit`'s value vector through the unembedding — its output-token bias.'''
    v = W_down[:, unit]                    # (D,) the fixed vector this memory writes
    scores = (v * norm_w) @ W_U.T          # (V,) push on every output token
    pos = torch.topk(scores, n_pos).indices.tolist()
    neg = torch.topk(-scores, n_neg).indices.tolist()
    print(f"unit {unit:>4}  ‖value‖ = {v.norm():.2f}")
    print(f"      promotes:  {[tok.decode([i]) for i in pos]}")
    print(f"      suppresses:{[tok.decode([i]) for i in neg]}")


for u in top_units:
    show_value(u)

unit 1396  ‖value‖ = 5.70
      promotes:  [' millions', ' million', ' thousand', 'million', ' thousands', ' Millions', ' lakh', ' Million']
      suppresses:['merce', 'lier', 'Intf', 'BLIC']
unit  571  ‖value‖ = 4.81
      promotes:  ['among', ' among', ' amongst', "\\''", "+'.", ']//', ' incarc', 'LINE']
      suppresses:['ritis', 'experience', 'urns', 'unei']
unit  577  ‖value‖ = 5.29
      promotes:  [' ', '1', '  ', '�', ' ninety', ' nineteen', ' eighteen', 'od']
      suppresses:[' TType', 'ζω', 'staking', 'toplasm']
unit 1211  ‖value‖ = 5.32
      promotes:  ['rowsiness', 'ught', 'arynx', ' OrderedDict', 'FTWARE', 'ilical', 'compan', '��']
      suppresses:[' point', ' points', 'point', 'points']
unit  896  ‖value‖ = 4.95
      promotes:  ['од', 'okers', 'OW', '~~~~~~~~~~~~~~~~', 'perms', 'burning', 'otta', 'ana']
      suppresses:[' jsonify', 'ceptives', ' slider', ' merchand']


**What you should see:**

- **unit 1396 promotes `' million'`, `' billion'`-adjacent magnitude words**: `' millions'`, `' million'`, `' thousand'`, `'million'`, `' thousands'`, `' Millions'`, `' lakh'`, `' Million'`. Every one of the top 8 is a magnitude word — including `' lakh'`, the South Asian unit for 100,000. This is the memory in full: **key = "big number after a currency cue" → value = "predict a magnitude word."**
- **unit 577 promotes `' '`, `'1'`, `' nineteen'`, `' eighteen'`** — consistent with its key ("a year is coming"), and note `'1'`, the leading digit of essentially every year in this corpus's era.
- **units 571, 896, 1211 promote apparent noise.** Their keys were interpretable but their values don't read as output tokens.

That last point is not a failure of the method — it's what the mechanism actually looks like. Most FFN values write **intermediate features** for later layers to consume, not output-token predictions. Only some memories, mostly in late layers, are wired directly to vocabulary. The logit lens can only see the latter.

## Does the memory generalize, or did we just describe the corpus?

The read of unit 1396 was reverse-engineered from 32 sentences, one of which contained `$4.2 billion`. A pattern-matcher for that one sentence is worthless; a **static rule** should fire on text it has never seen and stay quiet on superficially similar text that doesn't match.

So: held-out probes, split into ones that should fire (a number after a currency cue) and ones that shouldn't (a number with no currency cue). Note this is a real test — the key must key on *"currency context"*, not merely *"digit."*

In [8]:
UNIT = 1396

probes = [
    ("fire",  "Annual profits rose to $7.3"),
    ("fire",  "The startup raised $12"),
    ("fire",  "The bridge cost $890"),
    ("fire",  "Their debt reached $45"),
    ("quiet", "The temperature outside was 7.3"),
    ("quiet", "He waited for 12 minutes before"),
    ("quiet", "The recipe needs 3 cups of flour and"),
    ("quiet", "Chapter 7 of the book was"),
]

grab = {}
handle = model.model.layers[LAYER].mlp.down_proj.register_forward_pre_hook(
    lambda mod, args: grab.__setitem__("h", args[0].detach()))
print(f"unit {UNIT} @ layer {LAYER}, activation on the FINAL token of each held-out probe:\n")
with torch.no_grad():
    for expect, text in probes:
        model(**tok(text, return_tensors="pt"))
        print(f"   {grab['h'][0, -1, UNIT]:7.2f}   [{expect:5s}]  {text!r}")
handle.remove()

unit 1396 @ layer 25, activation on the FINAL token of each held-out probe:

     30.63   [fire ]  'Annual profits rose to $7.3'
     42.78   [fire ]  'The startup raised $12'
     13.17   [fire ]  'The bridge cost $890'
     27.48   [fire ]  'Their debt reached $45'
     -1.07   [quiet]  'The temperature outside was 7.3'
      0.17   [quiet]  'He waited for 12 minutes before'
      0.03   [quiet]  'The recipe needs 3 cups of flour and'
      0.14   [quiet]  'Chapter 7 of the book was'


**What you should see:** the four currency probes fire at **13 to 43**; the four numeric-but-not-currency probes sit between **-1.1 and 0.2** — indistinguishable from off. The gap is two orders of magnitude, on text the unit has never seen, and the model was never told about this distinction.

The key is genuinely keying on *"a monetary magnitude is in progress,"* not on the presence of a digit. And note what makes this a **memory** rather than a computation: the same rule, in the same fixed weights, applied to every input. Nothing about it adapts to the sentence.

## The causal test: ablate the memory

Correlation so far. The strong claim is **causal**: if this memory is what makes the model say "million," then deleting it should remove that prediction.

Zeroing `h[..., UNIT]` in the pre-hook removes exactly this unit's write and nothing else — the surgical version of "remove one row from the lookup table." (Returning a tuple from a `forward_pre_hook` *replaces* the module's inputs, which is how the edit takes effect.)

In [9]:
def next_token_probs(prompt, ablate=None, k=6):
    '''Next-token distribution, optionally with FFN unit `ablate` zeroed at LAYER.'''
    def hook(mod, args):
        h = args[0].clone()
        if ablate is not None:
            h[..., ablate] = 0.0          # delete this one memory's write
        return (h,)                        # returning a tuple replaces the module's inputs
    handle = model.model.layers[LAYER].mlp.down_proj.register_forward_pre_hook(hook)
    with torch.no_grad():
        logits = model(**tok(prompt, return_tensors="pt")).logits[0, -1]
    handle.remove()
    probs = logits.softmax(-1)
    top = [(tok.decode([i]), round(probs[i].item(), 3)) for i in torch.topk(probs, k).indices]
    return probs, top


WATCH = [" million", " billion", " thousand"]
watch_ids = [tok.encode(w)[0] for w in WATCH]

for prompt in ["The company reported revenue of $4.2",
               "Annual profits rose to $7.3",
               "Their debt reached $45"]:
    p_base, top_base = next_token_probs(prompt)
    p_abl,  top_abl  = next_token_probs(prompt, ablate=UNIT)
    print(f"{prompt!r}")
    print(f"   baseline top-6: {top_base}")
    print(f"   unit {UNIT} off: {top_abl}")
    for w, i in zip(WATCH, watch_ids):
        print(f"      p({w!r:11s}) {p_base[i]:.2e} -> {p_abl[i]:.2e}   ({p_abl[i]/p_base[i]:.2f}x)")
    print()

'The company reported revenue of $4.2'
   baseline top-6: [(' billion', 0.362), (' million', 0.158), ('5', 0.11), ('7', 0.047), ('4', 0.047), ('2', 0.045)]
   unit 1396 off: [(' billion', 0.189), ('5', 0.147), ('3', 0.08), ('6', 0.075), ('7', 0.073), ('2', 0.07)]
      p(' million' ) 1.58e-01 -> 1.74e-02   (0.11x)
      p(' billion' ) 3.62e-01 -> 1.89e-01   (0.52x)
      p(' thousand') 1.32e-05 -> 8.19e-06   (0.62x)

'Annual profits rose to $7.3'
   baseline top-6: [(' billion', 0.535), (' million', 0.095), ('5', 0.048), ('2', 0.038), ('3', 0.036), ('4', 0.035)]
   unit 1396 off: [(' billion', 0.226), ('3', 0.09), ('5', 0.086), ('0', 0.08), ('6', 0.073), ('2', 0.069)]
      p(' million' ) 9.54e-02 -> 1.41e-02   (0.15x)
      p(' billion' ) 5.35e-01 -> 2.26e-01   (0.42x)
      p(' thousand') 1.76e-05 -> 1.12e-05   (0.64x)

'Their debt reached $45'
   baseline top-6: [('0', 0.296), (' billion', 0.228), ('.', 0.091), (',', 0.083), (' million', 0.081), ('5', 0.034)]
   unit 1396 off: [('0'

**What you should see** — deleting one unit out of 1536, in one layer out of 30:

| prompt | `p(' million')` | `p(' billion')` |
|---|---|---|
| `revenue of $4.2` | 0.158 → 0.017 (**0.11x**) | 0.362 → 0.189 (0.52x) |
| `profits rose to $7.3` | 0.095 → 0.014 (**0.15x**) | 0.535 → 0.226 (0.42x) |
| `debt reached $45` | 0.081 → 0.036 (0.45x) | 0.228 → 0.167 (0.74x) |

`' million'` drops by up to **9x** and the freed probability mass flows to bare digits (`5`, `3`, `6`, `7`) — the model reverts to continuing the number instead of finishing the magnitude. That is as direct a demonstration of "this memory holds this behavior" as you can get with one line of code.

Two honest caveats. `' billion'` only halves, and survives as the top prediction: **the behavior is distributed** — other memories in other layers push the same way, so no single unit is load-bearing on its own. And the effect is weakest on `$45` (0.45x), where "45" could plausibly continue as `450`. One memory is a *contributor* to a behavior, not its sole owner. This is exactly why knowledge editing (ROME, MEMIT) edits *many* FFN weights at once rather than flipping a single unit.

## Why "memory" and not "computation": the write direction never changes

Here's the cleanest way to see what distinguishes the FFN from attention. Take one FFN memory and one attention head, and ask what each **writes into the residual stream** on two completely different inputs.

- The FFN unit writes `h_i · W_down[:, i]`. The input controls the scalar `h_i` — and *only* the scalar. The **direction** is a fixed column of a weight matrix.
- An attention head writes `W_O^h · (attention-weighted mix of values from other positions)`. Both the magnitude *and* the direction depend on what's in the context.

Compare with cosine similarity: for the FFN memory it should be exactly 1, for the head it should be near 0.

In [10]:
HEAD, D_h = 4, cfg.hidden_size // cfg.num_attention_heads
prompt_a = "The company reported revenue of $4.2"
prompt_b = "The mitochondria produce ATP, the energy currency of"


def writes(prompt):
    '''Return (this memory's write, this head's write) at the final token.'''
    g = {}
    h1 = model.model.layers[LAYER].mlp.down_proj.register_forward_pre_hook(
        lambda m, a: g.__setitem__("h", a[0].detach()[0, -1]))            # (d_ff,)
    h2 = model.model.layers[LAYER].self_attn.o_proj.register_forward_pre_hook(
        lambda m, a: g.__setitem__("heads", a[0].detach()[0, -1]))        # (D,) heads concatenated
    with torch.no_grad():
        model(**tok(prompt, return_tensors="pt"))
    h1.remove(); h2.remove()

    coef = g["h"][UNIT]
    ffn_write = coef * W_down[:, UNIT]                                    # scalar x fixed direction
    sl = slice(HEAD * D_h, (HEAD + 1) * D_h)                              # this head's slice of the concat
    W_O = model.model.layers[LAYER].self_attn.o_proj.weight               # (D, D)
    attn_write = W_O[:, sl] @ g["heads"][sl]                              # this head's write only
    return coef, ffn_write, attn_write


c_a, ffn_a, attn_a = writes(prompt_a)
c_b, ffn_b, attn_b = writes(prompt_b)

print(f"FFN memory {UNIT}:")
print(f"   activation h:  {c_a:8.2f}  vs {c_b:8.2f}    <- the input moves this")
print(f"   write norm:    {ffn_a.norm():8.2f}  vs {ffn_b.norm():8.2f}")
print(f"   cos(write_a, write_b) = {F.cosine_similarity(ffn_a, ffn_b, dim=0):.4f}   <- direction is FIXED")
print(f"\nAttention head {HEAD}:")
print(f"   write norm:    {attn_a.norm():8.2f}  vs {attn_b.norm():8.2f}")
print(f"   cos(write_a, write_b) = {F.cosine_similarity(attn_a, attn_b, dim=0):.4f}   <- direction is INPUT-DEPENDENT")

FFN memory 1396:
   activation h:     36.50  vs     0.31    <- the input moves this
   write norm:      207.93  vs     1.79
   cos(write_a, write_b) = 1.0000   <- direction is FIXED

Attention head 4:
   write norm:       18.75  vs     9.71
   cos(write_a, write_b) = -0.0156   <- direction is INPUT-DEPENDENT


**What you should see:** the FFN memory's activation swings from **36.5 to 0.31** and its write norm from **208 to 1.8** — but `cos = 1.0000`, exactly. Same direction, every input, always; only the volume knob moves. The attention head's write, on the same two inputs, has `cos ≈ -0.016` — an essentially unrelated direction.

That is the division of labor from [`../01_assembling_the_block.md`](../01_assembling_the_block.md), made numerical:

| | retrieves from | keys/values are | write direction |
|---|---|---|---|
| **attention** | the **sequence** (other positions) | computed fresh per forward pass | input-dependent |
| **FFN** | the **weights** (itself) | fixed learned parameters | fixed per unit |

Attention does dynamic routing over the current context. The FFN does static lookup over stored memories. And since the FFN holds ~75% of block parameters, most of what the model *knows* — as opposed to what it's currently *looking at* — lives in these key-value pairs.

## Where the framing gets fuzzy

The key-value story is a genuinely useful model, not a complete one. Three limits worth seeing rather than being told:

1. **Mid-layer values don't read as output tokens.** Look at a color-detector in layer 20 below: the key is beautifully clean, the value is unreadable through the unembedding — because it writes a feature for later layers, not a prediction.
2. **SwiGLU has two key matrices, not one.** `h_i = silu(gate_i · x) · (up_i · x)`. The gate decides *whether* to fire, `up` modulates *how much and with what sign*. "The key" is really a pair, which is why we read keys off the post-activation `h` (the actual coefficient on the value) rather than off `W_up` alone.
3. **Most units are polysemantic.** Unit 1396 is unusually clean. The typical unit fires on several unrelated things because the model packs more features than it has dimensions — superposition ([3.1/04](../../../part3_residual_connections_deep_networks/3.1_skip_connection/04_residual_stream_as_abstraction.md)). This is precisely the motivation for sparse autoencoders: find interpretable *directions* rather than hoping individual neurons are clean.

In [11]:
A20, ids20, lens20 = capture_ffn_acts(20, lines)
owner20 = [(li, p) for li, n in enumerate(lens20) for p in range(n)]

COLOR_UNIT = 882
col = A20[:, COLOR_UNIT]
print(f"layer 20, unit {COLOR_UNIT} — top-activating contexts (the key):")
for row in torch.topk(col, 5).indices.tolist():
    li, p = owner20[row]
    start = sum(lens20[:li])
    seq = ids20[start:start + lens20[li]]
    print(f"   {col[row]:6.2f}   ...{tok.decode(seq[max(0, p-9):p])}[[{tok.decode(seq[p:p+1])}]]")

v = model.model.layers[20].mlp.down_proj.weight[:, COLOR_UNIT]
scores = (v * norm_w) @ W_U.T
print(f"\nits value, through the unembedding (the value):")
print(f"   promotes: {[tok.decode([i]) for i in torch.topk(scores, 8).indices]}")

layer 20, unit 882 — top-activating contexts (the key):
     9.24   ... the grass is green, and ripe tomatoes are[[ red]]
     6.02   ...The sky is[[ blue]]
     2.12   ...The sky is blue, the grass is[[ green]]
     0.77   ...On Monday it[[ ra]]
     0.73   ...The[[ sky]]

its value, through the unembedding (the value):
   promotes: [' my', ' only', ' such', 'my', 'ATE', 'den', 'ate', 'ive']


**What you should see:** the key is unmistakable — the unit fires on `tomatoes are `**`red`**, `The sky is `**`blue`**, `the grass is `**`green`**. A color-completion detector, and it fires on the color token itself in a "X is COLOR" frame.

Its value promotes `' my'`, `' only'`, `' such'`, `'ATE'` — noise. Ten layers from the output, this memory's write is not a vote for a vocabulary item; it's a feature that layers 21-30 will read and transform. **Interpretable key, uninterpretable value** is the normal case in mid-stack, and a good reminder that the logit lens measures "how directly is this wired to the vocabulary," not "is this meaningful."

## Self-check

1. Why is `down_proj`'s *input* the right tensor to hook, rather than `gate_proj`'s or `up_proj`'s output?
2. On the `$4.2` token, one unit accounted for 81% of the FFN output *norm* but only 0.763 cosine to the full output. How can both be true, and which number better supports the "lookup table" framing?
3. Ablating unit 1396 cut `p(' million')` by 9x but only halved `p(' billion')`, which stayed the top prediction. What does that say about how behaviors map to units — and about single-neuron editing as a technique?
4. Why does the logit lens work better at layer 25 than at layer 20? What would you expect at layer 2?
5. We folded in `model.norm.weight` but not the normalization itself. Why can't we apply the actual RMSNorm to a single unit's value vector?
6. The FFN memory's write had `cos = 1.0000` across two unrelated inputs. Does that mean the FFN contributes the same thing to both? What *is* different?

### Answers

1. Because `h = silu(gate_proj(x)) * up_proj(x)` is the vector that actually multiplies the value vectors — the true per-memory coefficient. Either projection alone gives you half the gate: `gate` before its nonlinearity, or `up` without the gating. Hooking `down_proj`'s input also means you never reimplement the activation or the elementwise product, so you can't get it subtly wrong.
2. Norm and direction are different questions. One unit supplying 81% of the norm means it dominates *how far* the FFN moves the stream; cosine 0.763 means the ~1500 small writes still tilt *which way* it moves, because they're not aligned with the big one and partially cancel each other rather than canceling the dominant term. The cosine is the more honest number for the lookup-table claim: a few loud entries plus a quiet chorus, not one entry alone.
3. Behaviors are **distributed** across many units and layers — several memories push toward magnitude words, so removing one degrades the behavior without deleting it. `' million'` was more concentrated in this unit than `' billion'` was. The practical implication: single-neuron surgery is a good *diagnostic* (it isolates a contribution) but a poor *edit*. Real knowledge-editing methods (ROME, MEMIT) apply a rank-one or low-rank update to whole FFN matrices, often across several layers, precisely because the target behavior isn't stored in one place.
4. The logit lens pretends the remaining layers are the identity. At layer 25 of 30 only 5 layers remain, so a value vector's push on the vocabulary mostly survives to the output. At layer 20 there are 10 more layers of transformation, and mid-stack values are features for downstream consumption rather than token votes. At layer 2 you'd expect almost pure noise — with one systematic exception, since detokenization/copying circuits early on can leave values that look like the *input* token rather than a prediction.
5. RMSNorm divides by the RMS of the **whole** hidden vector: `x / sqrt(mean(x²)) * g`. That denominator is a property of the full residual stream at that position — every layer's accumulated contributions, not just this one unit's. A single value vector has no meaningful RMS of its own, and the true scale factor depends on inputs we're deliberately abstracting away. The learned per-dimension gain `g` *is* a fixed linear reweighting of the stream's axes, so folding that in is legitimate; the normalization isn't.
6. No — it contributes the same **direction** at wildly different **magnitudes** (norm 208 vs 1.8). The direction is a fixed weight column, so the only thing the input controls is how loudly this memory speaks. On the first input the memory is shouting; on the second it's effectively silent. "Same direction, input-dependent volume" is the whole signature of a static memory, and the contrast with attention (where direction itself moves) is what makes the FFN a lookup and attention a routing operation.

## Summary

What you established on a real 135M-parameter SwiGLU model:

- **The FFN decomposes exactly** into `out = Σ_i h_i · W_down[:, i]` — a sum of per-unit writes with no cross terms. This is what licenses talking about individual "memories" at all.
- **Firing is sparse and concentrated.** Fewer than 9% of memories are meaningfully active on a token, and a single one can carry 81% of the output norm.
- **Keys are readable from data.** Top-activating contexts told us unit 1396 detects "a monetary magnitude is in progress" — and it held up on held-out text (13-43 when firing vs ≈0 when not), keying on currency context rather than digits.
- **Values are readable through the unembedding**, in late layers. Unit 1396's value promotes `' million'`, `' billion'`, `' thousand'`, `' lakh'` — all eight top tokens are magnitude words.
- **The wiring is causal.** Zeroing that one unit cut `p(' million')` to 0.11x and pushed the mass back to bare digits.
- **The write direction is fixed** (`cos = 1.0000` across unrelated inputs) while an attention head's is not (`cos ≈ 0`). This is the sharpest available statement of the split: attention retrieves from the **sequence**, the FFN retrieves from the **weights**.

And the limits, equally worth carrying forward: behaviors are distributed (one unit is a contributor, not an owner), mid-layer values don't read as tokens, SwiGLU's key is a gate/up pair rather than a single row, and clean monosemantic units like 1396 are the exception — superposition is the rule, which is why interpretability work moved from neurons to learned sparse directions.

**Where this goes next:** the same decomposition is what makes Mixture-of-Experts natural (Part 7) — if the FFN is a lookup table over `d_ff` memories and only a few fire per token, then storing far more memories and routing to a subset is the obvious scaling move. MoE is this notebook's observation taken seriously.